<a href="https://colab.research.google.com/github/zeugirdoR/ricci-flow-tokenization/blob/main/ricci_tokenizer_a100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from collections import Counter
import time

def simple_wasserstein(p, q):
    return np.sum(np.abs(p - q))

def minimal_ricci_curvature(i, j, transition, alpha):
    n = transition.shape[0]
    mu_i = np.zeros(n)
    mu_i[i] = 1 - alpha
    mu_i += alpha * transition[i, :]
    mu_j = np.zeros(n)
    mu_j[j] = 1 - alpha
    mu_j += alpha * transition[j, :]
    W = simple_wasserstein(mu_i, mu_j)
    return 1.0 - W

def hybrid_ricci_tokenizer(text, num_merges, alpha, beta, verbose=True):
    if verbose:
        print(f"Corpus: {len(text)} chars, α={alpha}, β={beta}, merges={num_merges}")

    tokens = list(text.encode('utf-8'))
    vocab_size = 256
    merges = []
    start = time.time()

    for merge_num in range(num_merges):
        adjacency = np.zeros((vocab_size, vocab_size))
        for i in range(len(tokens) - 1):
            adjacency[tokens[i], tokens[i+1]] += 1

        row_sums = adjacency.sum(axis=1, keepdims=True)
        transition = adjacency / (row_sums + 1e-10)

        edges = []
        for i in range(vocab_size):
            for j in range(vocab_size):
                if adjacency[i, j] >= 2:
                    edges.append((i, j, adjacency[i, j]))

        if not edges:
            if verbose:
                print(f"Stopped at {merge_num} merges (no more edges)")
            break

        max_freq = max(e[2] for e in edges)
        scores = []
        for i, j, freq in edges:
            kappa = minimal_ricci_curvature(i, j, transition, alpha)
            kappa_norm = (kappa + 1) / 2
            freq_norm = freq / max_freq
            hybrid = beta * kappa_norm + (1 - beta) * freq_norm
            scores.append((i, j, hybrid, kappa, freq))

        scores.sort(key=lambda x: x[2], reverse=True)
        best_i, best_j, hybrid, kappa, freq = scores[0]

        try:
            token = bytes([best_i, best_j]).decode('utf-8', errors='replace')
        except:
            token = f"[{best_i},{best_j}]"

        merges.append((token, best_i, best_j, hybrid, kappa, freq))

        new_token = vocab_size
        vocab_size += 1

        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best_i and tokens[i+1] == best_j:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        if verbose and (merge_num + 1) % 500 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            elapsed = time.time() - start
            speed = (merge_num + 1) / elapsed
            print(f"  {merge_num+1:4d}: {comp:.2f}%, {speed:.1f} m/s")

    elapsed = time.time() - start
    final_comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100

    if verbose:
        print(f"✓ Done: {len(merges)} merges in {elapsed:.0f}s, {final_comp:.2f}%")

    return merges, tokens, final_comp

def simple_bpe(text, num_merges, verbose=True):
    if verbose:
        print(f"BPE: {len(text)} chars, {num_merges} merges")

    tokens = list(text.encode('utf-8'))
    merges = []
    start = time.time()

    for merge_num in range(num_merges):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)

        new_token = 256 + len(merges) - 1
        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best[0] and tokens[i+1] == best[1]:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        if verbose and (merge_num + 1) % 500 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            elapsed = time.time() - start
            speed = (merge_num + 1) / elapsed
            print(f"  {merge_num+1:4d}: {comp:.2f}%, {speed:.1f} m/s")

    elapsed = time.time() - start
    comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
    if verbose:
        print(f"✓ Done: {len(merges)} merges in {elapsed:.0f}s, {comp:.2f}%")
    return merges, tokens, comp

print("="*80)
print("A100 VALIDATION - Reproducing 80.18% Result")
print("="*80)
print()

import urllib.request
url = "https://www.gutenberg.org/files/11/11-0.txt"
with urllib.request.urlopen(url) as response:
    alice_full = response.read().decode('utf-8')

start_idx = alice_full.find("CHAPTER I")
alice = alice_full[start_idx:start_idx+150000]

split = int(len(alice) * 0.8)
train = alice[:split]
test = alice[split:]

print(f"Train: {len(train)} chars")
print(f"Test:  {len(test)} chars")
print()

print("="*80)
print("1. REPRODUCE CPU RESULT (4000 merges)")
print("="*80)
merges_r, tokens_train, comp_train = hybrid_ricci_tokenizer(train, 4000, 0.8, 0.2)

print()
print("Tokenizing test set...")
tokens_test = list(test.encode('utf-8'))
for idx, (_, b1, b2, _, _, _) in enumerate(merges_r):
    new_tok = 256 + idx
    i, new_tokens = 0, []
    while i < len(tokens_test):
        if i < len(tokens_test)-1 and tokens_test[i] == b1 and tokens_test[i+1] == b2:
            new_tokens.append(new_tok)
            i += 2
        else:
            new_tokens.append(tokens_test[i])
            i += 1
    tokens_test = new_tokens

test_comp = (1 - len(tokens_test) / len(test.encode('utf-8'))) * 100
print(f"Test compression: {test_comp:.2f}%")

print()
print("="*80)
print("2. BPE BASELINE (4000 merges)")
print("="*80)
merges_b, _, comp_bpe_train = simple_bpe(train, 4000)

tokens_test_bpe = list(test.encode('utf-8'))
for idx, pair in enumerate(merges_b):
    new_tok = 256 + idx
    i, new_tokens = 0, []
    while i < len(tokens_test_bpe):
        if i < len(tokens_test_bpe)-1 and tokens_test_bpe[i] == pair[0] and tokens_test_bpe[i+1] == pair[1]:
            new_tokens.append(new_tok)
            i += 2
        else:
            new_tokens.append(tokens_test_bpe[i])
            i += 1
    tokens_test_bpe = new_tokens

test_comp_bpe = (1 - len(tokens_test_bpe) / len(test.encode('utf-8'))) * 100
print(f"Test compression: {test_comp_bpe:.2f}%")

print()
print("="*80)
print("RESULTS")
print("="*80)
print()
print(f"Method           Train      Test       Improvement")
print("-"*60)
print(f"Hybrid Ricci     {comp_train:.2f}%    {test_comp:.2f}%")
print(f"BPE              {comp_bpe_train:.2f}%    {test_comp_bpe:.2f}%")
print(f"Difference       {comp_train-comp_bpe_train:+.2f}%    {test_comp-test_comp_bpe:+.2f}%")
print()

if test_comp > test_comp_bpe:
    print(f"🎉 VALIDATED! Ricci beats BPE on held-out data!")
    print(f"   Test improvement: +{test_comp-test_comp_bpe:.2f}%")
elif abs(test_comp - test_comp_bpe) < 0.5:
    print(f"✓ Essentially tied on test set")
else:
    print(f"Note: BPE ahead on test by {test_comp_bpe-test_comp:.2f}%")

print()
print("="*80)
print("3. TOP 20 TOKENS COMPARISON")
print("="*80)
print()
print("Ricci tokens:")
for i, (tok, _, _, score, kappa, freq) in enumerate(merges_r[:20], 1):
    print(f"  {i:2d}. '{tok:>4s}' (score={score:.3f}, freq={freq:.0f})")

print()
print("BPE tokens:")
for i, pair in enumerate(merges_b[:20], 1):
    try:
        tok = bytes(pair).decode('utf-8', errors='replace')
    except:
        tok = f"[{pair[0]},{pair[1]}]"
    print(f"  {i:2d}. '{tok:>4s}'")

A100 VALIDATION - Reproducing 80.18% Result

Train: 115623 chars
Test:  28906 chars

1. REPRODUCE CPU RESULT (4000 merges)
Corpus: 115623 chars, α=0.8, β=0.2, merges=4000
   500: 62.55%, 5.6 m/s
  1000: 69.43%, 3.8 m/s
  1500: 72.97%, 2.7 m/s
  2000: 75.31%, 2.0 m/s
  2500: 77.07%, 1.5 m/s
  3000: 78.40%, 1.1 m/s
  3500: 79.65%, 0.9 m/s
  4000: 80.48%, 0.7 m/s
✓ Done: 4000 merges in 5423s, 80.48%

Tokenizing test set...
Test compression: 76.09%

2. BPE BASELINE (4000 merges)
BPE: 115623 chars, 4000 merges
   500: 62.39%, 31.1 m/s
  1000: 69.27%, 35.5 m/s
  1500: 72.78%, 38.4 m/s
  2000: 75.13%, 40.7 m/s
  2500: 76.89%, 42.7 m/s
  3000: 78.25%, 44.3 m/s
  3500: 79.49%, 45.8 m/s
  4000: 80.36%, 47.3 m/s
✓ Done: 4000 merges in 85s, 80.36%
Test compression: 75.87%

RESULTS

Method           Train      Test       Improvement
------------------------------------------------------------
Hybrid Ricci     80.48%    76.09%
BPE              80.36%    75.87%
Difference       +0.12%    +0.21%

🎉 VA